<a href="https://colab.research.google.com/github/SunnyFoxxy/stock-market-analytics-zoomcamp-2026/blob/main/Homework_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import requests
import io
import yfinance as yf
import logging

logging.getLogger('yfinance').setLevel(logging.CRITICAL)

Q1. What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value? (2 points)

In [17]:
url = "https://www.iposcoop.com/ipos-recently-filed/"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
print(resp.status_code)
tables = pd.read_html(io.StringIO(resp.text))
print(len(tables))
ipos = tables[0]

ipos.columns
ipos['Expected To Trade'].value_counts()
withdrawn = ipos[ipos['Expected To Trade'] == 'Withdrawn'].copy()
len(withdrawn)
withdrawn['File Date'] = pd.to_datetime(withdrawn['File Date'])
withdrawn = withdrawn[withdrawn['File Date'] < '2026-09-11'].copy()
len(withdrawn)


200
1


31

In [18]:
def classify(name):
    if "Technologies" in name:
        return "Technologies"
    elif "Acquisition Corp" in name or "Acquisition Corporation" in name or "Corp" in name:
        return "Acquisition Corp"
    elif "Inc" in name or "Incorporated" in name:
        return "Inc."
    elif "Group" in name:
        return "Group"
    elif "Ltd" in name or "Limited" in name:
        return "Limited"
    elif "Holdings" in name or "Holding" in name:
        return "Holdings"
    else:
        return "Other"

In [19]:

withdrawn['Company Type'] = withdrawn['Company'].apply(classify)
withdrawn['Company Type'].value_counts()

,count
Company Type,
Other,8
Limited,7
Acquisition Corp,5
Holdings,4
Technologies,3
Group,3
Inc.,1


In [20]:

def parse_price(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace('$', '').replace(',', '').strip()
    if x in ('-', ''):
        return np.nan
    try:
        return float(x)
    except ValueError:
        return np.nan



In [21]:

withdrawn['Price Low num'] = withdrawn['Price Low'].apply(parse_price)
withdrawn['Price High num'] = withdrawn['Price High'].apply(parse_price)
withdrawn['Avg_price'] = (withdrawn['Price Low num'] + withdrawn['Price High num']) / 2

withdrawn['Shares (millions)'] = withdrawn['Shares (millions)'].apply(parse_price)
withdrawn['Est $ Vol (millions)'] = withdrawn['Est $ Vol (millions)'].apply(parse_price)


calc = withdrawn['Shares (millions)'] * withdrawn['Avg_price']
withdrawn['Shares_offered_value'] = calc.fillna(withdrawn['Est $ Vol (millions)'])


result = withdrawn.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)



In [22]:

result




,Shares_offered_value
Company Type,
Acquisition Corp,499.9850
Inc.,351.0000
Holdings,311.6575
Other,290.4450
Limited,197.2500
Technologies,184.9000
Group,32.5000


Q2. What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?

In [23]:
url25 = "https://www.iposcoop.com/2025-pricings/"
resp = requests.get(url25, headers=headers)
ipos25 = pd.read_html(io.StringIO(resp.text))[0]
print(len(ipos25))
print(ipos25.columns.tolist())


231
['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)', 'Offer Price', '1st Day Close', 'Current Price', 'Return', 'SCOOP Rating']


In [24]:
ipos25['Offer Date'] = pd.to_datetime(ipos25['Offer Date'])
ipos25['Return_num'] = ipos25['Return'].apply(lambda x: parse_price(str(x).replace('%', '')))


filtered25 = ipos25[(ipos25['Offer Date'] < '2025-09-01') & (ipos25['Return_num'] != 0)].copy()
print(len(filtered25))
tickers = filtered25['Symbol'].tolist()


146


In [25]:
stocks = []
for t in tickers:
    try:
        df = yf.download(t, start='2025-01-01', end='2026-09-12', progress=False, auto_adjust=False)
    except Exception as e:
        print(t, 'error:', e)
        continue
    if df.empty:
        continue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)


    df['Ticker'] = t
    df['growth_252d'] = df['Close'] / df['Close'].shift(252)
    df['volatility'] = df['Close'].rolling(30).std() * np.sqrt(252)
    df['Sharpe'] = (df['growth_252d'] - 0.05) / df['volatility']
    stocks.append(df)


stocks_df = pd.concat(stocks)
print(stocks_df['Ticker'].nunique())

131


In [26]:
day = stocks_df.loc['2026-09-11']
day[['growth_252d', 'volatility', 'Sharpe']].describe()


Price,growth_252d,volatility,Sharpe
count,129.000000,130.000000,129.000000
mean,1.059676,27.781138,inf
std,3.059882,67.083691,NaN
min,0.001005,0.000000,-0.040147
25%,0.149038,2.325895,0.011942
50%,0.596026,8.122327,0.048009
75%,1.040233,24.019924,0.134248
max,33.638270,627.698106,inf


Q3. What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?

In [27]:
for m in range(1, 13):
    stocks_df[f'future_growth_{m}_m'] = (
        stocks_df.groupby('Ticker')['Close'].shift(-21 * m) / stocks_df['Close']
    )
stocks_df[stocks_df['Ticker'] == tickers[0]][['Close', 'future_growth_1_m']].tail(25)


for m in range(1, 13):
    stocks_df[f'future_growth_{m}_m'] = (
        stocks_df.groupby('Ticker')['Close'].shift(-21 * m) / stocks_df['Close']
    )

stocks_q3 = stocks_df.reset_index()
min_dates = (stocks_q3.groupby('Ticker')['Date'].min()
             .reset_index()
             .rename(columns={'Date': 'min_date'}))

first_day = stocks_q3.merge(
    min_dates,
    left_on=['Ticker', 'Date'],
    right_on=['Ticker', 'min_date'],
    how='inner'
)
print(len(first_day))


growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
stats = first_day[growth_cols].describe()
stats


medians = stats.loc['50%']
print(medians.idxmax(), round(medians.max(), 4))


131
future_growth_1_m 0.9306


Q4. What is the total profit (in {$} thousands) you  would  have  earned  by investing {$} 1000 every time a stock was oversold (RSI < 30)?

In [28]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")
print(df.shape)
print([c for c in df.columns if 'rsi' in c.lower() or 'growth_future' in c.lower()][:20])

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=42dee2a6-0407-402e-8684-7c21afbe3339
To: /content/data.parquet
100%|██████████| 130M/130M [00:01<00:00, 88.9MB/s]


(229932, 203)
['growth_future_30d', 'rsi', 'fastk_rsi', 'fastd_rsi', 'cdl3starsinsouth']


In [29]:

print('Date' in df.columns, df.index.name)

df['Date'] = pd.to_datetime(df['Date'])

selected_df = df[
    (df['rsi'] < 30) &
    (df['Date'] >= '2000-01-01') &
    (df['Date'] <= '2025-06-01')
]

print('Quantity:', len(selected_df))
print('Growth future:', selected_df['growth_future_30d'].mean())
print('Win rate:', (selected_df['growth_future_30d'] > 1).mean())

net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()
print('Net income, $K:', round(net_income / 1000, 1))


True None
Quantity: 5206
Growth future: 1.0126403360751977
Win rate: 0.5512869765655013
Net income, $K: 65.8


Q5. How would you change the strategy if you want to increase profitability?

Wait for the post-IPO correction instead of buying at the peak of the hype. Our Q3 results show that median returns are highest in the first month and decline afterward, meaning prices tend to drift down as the initial excitement fades. A better entry point could be after the lock-up period expires (typically 90–180 days after the IPO), when insiders are allowed to sell and the resulting selling pressure often pushes the price lower. Buying after this pullback lowers the entry price and filters out stocks that only rose on hype.
Or build a dataset of past IPOs with features such as sector, offering size, where the final IPO price landed relative to the initial price range, first-day return, early volatility, and overall market conditions (e.g., whether the S&P 500 is above its 200-day moving average). Then train a classifier to predict whether the stock's growth after N months will be greater than 1 (i.e., a positive return). The strategy would only buy IPOs where the model's predicted probability of a positive return is high, filtering out the majority of IPOs that tend to decline.